In [1]:
%pip install torch torchvision torchaudio

  Obtaining dependency information for torchvision from https://files.pythonhosted.org/packages/1c/a9/c272623a0f735c35f0f6cd6dc74784d4f970e800cf063bb76687895a2ab9/torchvision-0.26.0-cp312-cp312-win_amd64.whl.metadata
  Obtaining dependency information for torchaudio from https://files.pythonhosted.org/packages/23/a8/941277ecc39f7a0a169d554302a1f1afd87c1d94a8aec828891916cea59a/torchaudio-2.11.0-cp312-cp312-win_amd64.whl.metadata
   ---------------------------------------- 0.0/4.3 MB ? eta -:--:--
    --------------------------------------- 0.1/4.3 MB 1.7 MB/s eta 0:00:03
   ----- ---------------------------------- 0.6/4.3 MB 8.0 MB/s eta 0:00:01
   ------------ --------------------------- 1.3/4.3 MB 10.2 MB/s eta 0:00:01
   ----------------- ---------------------- 1.9/4.3 MB 11.0 MB/s eta 0:00:01
   ----------------------- ---------------- 2.5/4.3 MB 11.3 MB/s eta 0:00:01
   ---------------------------- ----------- 3.1/4.3 MB 11.5 MB/s eta 0:00:01
   ----------------------------------- 


[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import torch
import torch.nn as nn
import torch.optim as optim

# ==============================
# 1. Dataset Simples
# Labels: 0 => Sentimento Ruim / 1 => Sentimento Bom
# ==============================

sentences = [
("Eu amo esse filme", 1),
("Esse filme é ótimo", 1),
("Experiência Incrível", 1),
("Eu odeio esse filme", 0),
("Esse filme é terrível", 0),
("Experiência horrivel", 0),
]

In [6]:
# Função básica para separar a frase em palavras minúsculas
def tokenize(sentence):
    return sentence.lower().split()

# Construindo vocabulário com token de segurança para palavras desconhecidas
UNK_TOKEN = 1
vocab = {"<UNK>": UNK_TOKEN}

# Preenchendo o vocabulário com as palavras do dataset de treino
for sentence, _ in sentences:
    for word in tokenize(sentence):
        if word not in vocab:
            vocab[word] = len(vocab) + 1 # IDs começam do 2 em diante

# Tamanho do vocabulário (+1 para considerar o 0 que será usado no Padding)
vocab_size = len(vocab) + 1

def encode(sentence):
    # Se a palavra existir no vocab, retorna o ID dela. Se não, retorna o UNK_TOKEN (1)
    return [vocab.get(word, UNK_TOKEN) for word in tokenize(sentence)]

encoded_data = [(encode(s), label) for s, label in sentences]

# Função para aplicar Padding (adicionar zeros)
def pad_sequence(seq, max_len):
    return seq + [0] * (max_len - len(seq))

# Descobrimos qual é a frease mais longa do nosso dataset
max_len = max(len(seq) for seq, _ in encoded_data)

# Criando os tensores finais (X = dados de entrada, y = rótulos/respostas)
X = torch.tensor([pad_sequence(seq, max_len) for seq, _ in encoded_data])
y = torch.tensor([label for _, label in encoded_data], dtype=torch.float32)

print("Tensores: ")

print("=" *50)
print("Entradas: ")
print(X)

print("=" *50)
print("Rótulos: ")
print(y)

Tensores: 
Entradas: 
tensor([[ 2,  3,  4,  5],
        [ 4,  5,  6,  7],
        [ 8,  9,  0,  0],
        [ 2, 10,  4,  5],
        [ 4,  5,  6, 11],
        [ 8, 12,  0,  0]])
Rótulos: 
tensor([1., 1., 1., 0., 0., 0.])


In [9]:
class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(SentimentRNN, self).__init__()
        
        # Camada que mapeia palavras para vetores
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # A Vanilla RNN. batch_firs=True indica que o tensor de entrada
        # terá o formato (tamano_do_lote, tamanho_da_sequencia, dimensão_do_embedding)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        
        #Camada de classificação final
        self.fc = nn.Linear(hidden_dim, 1)
        
        # Função de ativação para gerar uma probabilidade (0 a 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        # 1. Passa as palavras pela camada de Embedding
        x = self.embedding(x)
    
        # 2. Passa pela RNN. Uma RNN simples retorna as saídas de todos os passos
        # e o último estado oculto (hidden_state).
        out_rnn, hidden = self.rnn(x)

        # 3. Queremos apenas o último estado oculto (-1) para a classificação
        out = self.fc(hidden[-1])

        # 4. Retorna a probabilidade
        return self.sigmoid(out)

# Instanciando nosso modelo com hiperparâmetros de exemplo
embedding_dim = 10
hidden_dim = 16

model = SentimentRNN(vocab_size, embedding_dim, hidden_dim)

In [11]:
criterion = nn.BCELoss() # Função de perda para classificação binária
optimizer = optim.Adam(model.parameters(), lr=0.01) # Algoritimo de otimização

epochs = 100

for epochs in range(epochs):
    # Coloca o modelo em modo de treinamento
    model.train()
    
    # Foward pass: passa os dados pelo modelo
    outputs = model(X).squeeze()
    loss = criterion(outputs, y)
    
    #Backward pass: calcula os gradientes e atualiza os pesos
    optimizer.zero_grad() # Zera gradientes antigos
    loss.backward() # Calcula o erro de trás para frente
    optimizer.step() # Atualiza os pesos de rede
    
    # Imprime o progresso a cada 10 épocas
if (epochs +1) % 10 == 0:
    print(f"Epoch {epochs+1}, Loss: {loss.item() :.5f}")

Epoch 100, Loss: 0.00001


In [12]:
def predict(sentence):
    # Coloca o modelo em modo de avaliação
    model.eval()

    # Pré-processamento idêntico ao treino
    seq = encode(sentence)
    seq = pad_sequence(seq, max_len)
    seq = torch.tensor([seq]) # Adiciona dimensão de batch artificial

    # Desativa cálculo de gradientes
    with torch.no_grad():
        output = model(seq).item()

    # Define a fronteira de decisão: maior que 0.5 é Positivo, senão Negativo
    return "Positivo" if output > 0.5 else "Negativo"

In [13]:
# Testando com frases conhecidas e desconhecidas
print("\nTestes:")
print("Eu amo esse filme ->", predict("Eu amo esse filme"))
print("Esse filme é terrível ->", predict("Esse filme é terrível"))

# Teste do OOV (palavras não conhecidas pelo vocabulário)
print("O filme do Silvio Santos é horrível ->", predict("O filme do Silvio Santos é horrível"))

# print("O filme do Silvio Santos e horrível", predict("O filme do Silvio Santos é horrivel"))


Testes:
Eu amo esse filme -> Positivo
Esse filme é terrível -> Negativo
O filme do Silvio Santos é horrível -> Negativo
